# 📊 Exploratory Data Analysis: Statistical Functions in Python
**Course**: Data Science Class | **Session**: Statistical Functions in EDA  
**Dataset**: `student_learning_performance_eda_150.csv` (150 students)  

---

### 🎯 Learning Objectives for Today
1. Compute and interpret descriptive summary statistics using `describe()`.
2. Understand central tendency (**mean**, **median**, **mode**) and know when each is misleading.
3. Measure data dispersion using **variance** and **standard deviation**.
4. Calculate **percentiles**, **quartiles**, and **Interquartile Range (IQR)**.
5. Detect anomalies using Tukey’s 1.5 $\times$ IQR rule and boxplots.
6. Identify **skewness** and distribution shapes.
7. Perform multi-group comparative analysis with `groupby()` and `agg()`.
8. Analyze relationships with **covariance** and **correlation**.

## 0. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display and style options
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid", palette="muted")

# Load dataset
df = pd.read_csv('student_learning_performance_eda_150.csv')
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

---
## Part 1: Quick Statistical Overview (`describe()`)

`df.describe()` generates the 5-number summary (min, Q1, median, Q3, max) plus count, mean, and standard deviation.

In [ ]:
# Overall statistical summary of all numerical columns
df.describe().T

💡 **Discussion Question for the Class**:
- Look at `study_hours_per_week`: Mean is ~18.3, Median (50%) is ~17.0, Max is 58.0.
- What does this difference between mean and median hint at?

---
## Part 2: Measures of Central Tendency & Dispersion
- **Mean**: Sum of all values divided by total count (sensitive to outliers).
- **Median**: Middle value when sorted (robust to outliers).
- **Mode**: Most frequent value(s) (useful for categorical & discrete counts).
- **Variance ($s^2$)**: Average squared deviation from the mean.
- **Standard Deviation ($s$)**: Square root of variance; expressed in the *original units*.

In [ ]:
col = 'final_exam_score'

mean_val = df[col].mean()
median_val = df[col].median()
mode_val = df['assignments_completed'].mode()[0] # Mode works great on discrete counts
variance_val = df[col].var()
std_val = df[col].std()

print(f"{col} - Mean: {mean_val:.2f}")
print(f"{col} - Median: {median_val:.2f}")
print(f"assignments_completed - Mode: {mode_val}")
print(f"{col} - Variance: {variance_val:.2f}")
print(f"{col} - Standard Deviation: {std_val:.2f}")

### 🧪 In-Class Exercise 1 (Students)
1. Calculate the mean, median, and standard deviation for `attendance_rate`.
2. Compare the mean and median. Is the mean higher or lower than the median?
3. What does that tell you about how most students attend class?

In [ ]:
# TODO: Student solution here
# mean_att = ...
# median_att = ...
# std_att = ...


---
## Part 3: Skewness & Distribution Shape
- **Symmetric** (Skew $\approx 0$): Mean $\approx$ Median
- **Right-Skewed / Positive** (Skew $> 0$): Long tail on right, Mean $>$ Median (pulled by high values)
- **Left-Skewed / Negative** (Skew $< 0$): Long tail on left, Mean $<$ Median (pulled by low values)

In [ ]:
# Check skewness across numeric features
print("Skewness values:")
print(df.select_dtypes(include=np.number).skew().round(3))

# Visualizing distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Right-skewed example
sns.histplot(df['study_hours_per_week'], kde=True, ax=axes[0], color='teal')
axes[0].axvline(df['study_hours_per_week'].mean(), color='red', linestyle='--', label=f"Mean: {df['study_hours_per_week'].mean():.1f}")
axes[0].axvline(df['study_hours_per_week'].median(), color='black', linestyle='-', label=f"Median: {df['study_hours_per_week'].median():.1f}")
axes[0].set_title("Right-Skewed: Study Hours per Week")
axes[0].legend()

# Left-skewed example
sns.histplot(df['attendance_rate'], kde=True, ax=axes[1], color='coral')
axes[1].axvline(df['attendance_rate'].mean(), color='red', linestyle='--', label=f"Mean: {df['attendance_rate'].mean():.1f}")
axes[1].axvline(df['attendance_rate'].median(), color='black', linestyle='-', label=f"Median: {df['attendance_rate'].median():.1f}")
axes[1].set_title("Left-Skewed: Attendance Rate")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Part 4: Percentiles, Quartiles & Outlier Detection

### The 1.5 $\times$ IQR Rule (Tukey's Fence)
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}$$
$$\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$$

In [ ]:
# Example on study_hours_per_week
target_col = 'study_hours_per_week'

q1 = df[target_col].quantile(0.25)
q3 = df[target_col].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print(f"Q1 (25th percentile): {q1:.2f}")
print(f"Q3 (75th percentile): {q3:.2f}")
print(f"IQR: {iqr:.2f}")
print(f"Lower Bound: {lower_bound:.2f}")
print(f"Upper Bound: {upper_bound:.2f}")

# Find outliers
outliers = df[(df[target_col] < lower_bound) | (df[target_col] > upper_bound)]
print(f"\nFound {len(outliers)} outlier(s) in {target_col}:")
outliers[['student_id', target_col, 'final_exam_score']]

In [ ]:
# Visualize with a Boxplot
plt.figure(figsize=(8, 3))
sns.boxplot(x=df[target_col], color='lightgreen', flierprops=dict(marker='o', markersize=8, markerfacecolor='red'))
plt.title(f"Boxplot of {target_col} with Outliers in Red")
plt.show()

### 🧪 In-Class Exercise 2 (Students)
Detect outliers in `attendance_rate`:
1. Compute Q1, Q3, and IQR for `attendance_rate`.
2. Calculate the lower and upper bounds using the 1.5 $\times$ IQR rule.
3. Filter the dataframe to identify which students fall below the lower bound.
4. How do their `final_exam_score` values look?

In [ ]:
# TODO: Student solution here
# q1_att = ...
# q3_att = ...
# iqr_att = ...
# lower_att = ...
# attendance_outliers = ...


---
## Part 5: Comparative Statistics with `groupby()` + `agg()`

Aggregate functions allow us to slice numerical metrics across categorical groups.

In [ ]:
# Comparing final exam score by study group participation
group_stats = df.groupby('study_group')['final_exam_score'].agg(
    count='count',
    mean='mean',
    median='median',
    std='std',
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25)
).round(2)

group_stats

In [ ]:
# Multi-dimensional aggregation: Group by study_group and prep_course
multi_group = df.groupby(['study_group', 'prep_course'])['final_exam_score'].agg([
    'count', 'mean', 'median', 'std'
]).round(2)

multi_group

### 🧪 In-Class Exercise 3 (Students)
Use `groupby()` and `agg()` to answer:
1. Group by `prep_course` and compute the mean and median for BOTH `study_hours_per_week` and `final_exam_score`.
2. Did students who took the prep course study more hours, or did they just score better?

In [ ]:
# TODO: Student solution here
# df.groupby('prep_course').agg(...)


---
## Part 6: Relationships - Covariance & Correlation
- **Covariance**: Measures joint variability. Large positive values mean variables increase together, but scale depends on units ($Cov(X,Y)$ is hard to interpret alone).
- **Pearson Correlation ($r$)**: Standardized covariance between -1 and +1.
  - $+1.0$: Perfect positive linear relationship
  - $0.0$: No linear relationship
  - $-1.0$: Perfect negative linear relationship

In [ ]:
# Covariance between study hours and final score
cov_val = df['study_hours_per_week'].cov(df['final_exam_score'])
corr_val = df['study_hours_per_week'].corr(df['final_exam_score'])

print(f"Covariance: {cov_val:.2f} (Units: hours * exam points)")
print(f"Correlation: {corr_val:.2f} (Scale-independent -1 to +1)")

In [ ]:
# Full Correlation Matrix
numeric_df = df.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr().round(2)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)
plt.title("Correlation Heatmap")
plt.show()

### 🧪 In-Class Challenge (Individual or Pairs - 10 mins)
1. Which variable has the **strongest correlation** with `final_exam_score`?
2. What is the correlation between `sleep_hours` and `final_exam_score`? Does sleeping more guarantee higher scores?
3. **Discussion Question**: "Correlation does not imply causation." How would you explain this in the context of this dataset to a school principal?

In [ ]:
# TODO: Find the correlations sorted by relationship to final_exam_score
corr_matrix['final_exam_score'].sort_values(ascending=False)